In [1]:
#Cargar el documento de los chunks
import json
from langchain_core.documents import Document

with open("documento.json", "r", encoding="utf-8") as f:
    data = json.load(f)

documents = []

for item in data:
    documents.append(
        Document(
            page_content=item["content"],
            metadata={"title": item["title"]}
        )
    )

print("Documentos cargados:", len(documents))

Documentos cargados: 19


In [2]:
#imports mas configs

# Configuración inicial
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import os
import time
from dotenv import load_dotenv

load_dotenv()

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.callbacks.base import BaseCallbackHandler

from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_classic.evaluation import load_evaluator

class SlowStreamingHandler(BaseCallbackHandler):
    def on_llm_new_token(self, token: str, **kwargs) -> None:
        print(token, end="", flush=True)
        time.sleep(0.03)  # 👈 ajusta aquí la velocidad

stream_handler = SlowStreamingHandler()

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.8,
    streaming=True,
    request_timeout=600,
    callbacks=[stream_handler]
)

store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [3]:
# --- SPLITTER ---
text_splitter = CharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

# --- EMBEDDINGS ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# --- VECTOR STORE ---
vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever()

# --- PROMPT ---
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "Responde preguntas sobre 3M Tours usando la información disponible. "
     "Explica de forma clara y un poco detallada, agregando contexto útil para el usuario."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{context}\n{input}")
])

In [ ]:
# --- RAG ---
document_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, document_chain)

# --- CHATBOT CON MEMORIA ---
chatbot = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# --- PRUEBA ---
response = chatbot.invoke(
    {"input": "¿Qué es 3M Tours?"},
    config={"configurable": {"session_id": "1"}}
)

print("\nRespuesta:", response.get("answer", response))

# --- CHAT CONTINUO ---
while True:
    pregunta = input("\nTú: ")

    response = chatbot.invoke(
        {"input": pregunta},
        config={"configurable": {"session_id": "1"}}
    )

    print("\nBot:", response.get("answer", response))

3M Tours es una empresa de turismo digital fundada en 2023 en Puerto Montt, Chile, que opera como un marketplace de servicios turísticos. Su plataforma digital, accesible desde una aplicación móvil o web, permite a los usuarios explorar, comparar y reservar una variedad de experiencias de viaje, como tours, actividades y alojamientos. 

La propuesta principal de 3M Tours es integrar múltiples servicios turísticos en una sola interfaz intuitiva y confiable, facilitando la planificación completa de viajes en el sur de Chile. La empresa se posiciona como un actor clave para mejorar la experiencia de los viajeros en esta región, ofreciendo soluciones prácticas y centralizadas.

Con una visión a largo plazo, 3M Tours aspira a convertirse en la principal plataforma de turismo inteligente en el sur de Sudamérica para el año 2030, destacándose por su innovación tecnológica y la calidad de su servicio.

Error in RootListenersTracer.on_chain_end callback: KeyError('output')



Respuesta: 3M Tours es una empresa de turismo digital fundada en 2023 en Puerto Montt, Chile, que opera como un marketplace de servicios turísticos. Su plataforma digital, accesible desde una aplicación móvil o web, permite a los usuarios explorar, comparar y reservar una variedad de experiencias de viaje, como tours, actividades y alojamientos. 

La propuesta principal de 3M Tours es integrar múltiples servicios turísticos en una sola interfaz intuitiva y confiable, facilitando la planificación completa de viajes en el sur de Chile. La empresa se posiciona como un actor clave para mejorar la experiencia de los viajeros en esta región, ofreciendo soluciones prácticas y centralizadas.

Con una visión a largo plazo, 3M Tours aspira a convertirse en la principal plataforma de turismo inteligente en el sur de Sudamérica para el año 2030, destacándose por su innovación tecnológica y la calidad de su servicio.
¡Gracias por compartir información sobre 3M Tours! A continuación, responderé tus

Error in RootListenersTracer.on_chain_end callback: KeyError('output')



Bot: ¡Gracias por compartir información sobre 3M Tours! A continuación, responderé tus posibles preguntas basándome en los detalles proporcionados.

### ¿Qué problema resuelve 3M Tours?
3M Tours aborda un problema común en la región de Los Lagos, Chile: la fragmentación de la oferta turística. Dado que la región es un destino popular para turistas por su belleza natural y cultura, la falta de una plataforma eficiente para encontrar, comparar y reservar servicios de manera centralizada puede ser un obstáculo para los visitantes. 3M Tours resuelve este problema al reunir en un solo lugar información sobre tours, alojamientos y actividades, permitiendo a los viajeros planificar y reservar sus experiencias de manera fácil y eficiente.

### ¿Qué es 3M Tours y dónde opera?
3M Tours es una plataforma digital de turismo inteligente que facilita la planificación de viajes en el sur de Chile, específicamente en la región de Los Lagos. Su base de operaciones está en Puerto Montt, una ciudad que 